# Figuring out the dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Define Local Path

In [2]:
import os

# TODO: Fill in the Google Drive path where you uploaded the CW_folder_UG
# Example: GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'Colab Notebooks/Computer Vision/CW_folder_UG'

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'Colab Notebooks/CW_Folder_UG'
GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print(os.listdir(GOOGLE_DRIVE_PATH))

['CW_Dataset', 'Models', 'Personal_Dataset', 'Code', 'test_function.ipynb', 'work.ipynb']


## UnZip Dataset and load to drive


In [3]:
# Identify path to zipped dataset
zip_path = os.path.join(GOOGLE_DRIVE_PATH, 'CW_Dataset/CW_Dataset.zip')

# Copy it to Colab
!cp '{zip_path}' .

# Unzip it
!yes|unzip -q CW_Dataset.zip

# Delete zipped version from Colab (not from Drive)
!rm CW_Dataset.zip

In [4]:
import os

print(os.listdir())
print(os.listdir("CW_Dataset"))
print(os.listdir("CW_Dataset/train")[:5])
print(os.listdir("CW_Dataset/test")[:5])

['.config', 'drive', 'CW_Dataset', 'sample_data']
['README.txt', 'train', 'test']
['train_9513.jpg', 'train_3119.jpg', 'train_11010.jpg', 'train_6998.jpg', 'train_1914.jpg']
['test_0210.jpg', 'test_0612.jpg', 'test_0819.jpg', 'test_0835.jpg', 'test_0846.jpg']


## Discover Image sizes


In [5]:
import os
import cv2

DATASET_PATH = '/content/CW_Dataset'

def analyse_image_sizes(split):
  folder = os.path.join(DATASET_PATH, split)

  sizes = []
  failed = 0

  for file in os.listdir(folder):
    if file.endswith(".jpg"):
      path = os.path.join(folder, file)

      img = cv2.imread(path)
      if img is None:
        failed += 1
        continue

      h, w = img.shape[:2]
      sizes.append((w,h))
  return sizes, failed

def print_stats(name, sizes, failed):
  print(f"\n==== {name} Set ====")
  print(f"Total images: {len(sizes)}")
  print(f"Failed {failed}")

  widths = [s[0] for s in sizes]
  heights = [s[1] for s in sizes]

  print("\n Min & MaX")
  print(f"Width: {min(widths)} -> {max(widths)}")
  print(f"Height: {min(heights)} -> {max(heights)}")

train_sizes, train_failed = analyse_image_sizes("train")
test_sizes, test_failed = analyse_image_sizes("test")


print_stats("train", train_sizes, train_failed)
print_stats("test", test_sizes, test_failed)


==== train Set ====
Total images: 13300
Failed 0

 Min & MaX
Width: 100 -> 200
Height: 100 -> 200

==== test Set ====
Total images: 850
Failed 0

 Min & MaX
Width: 100 -> 200
Height: 100 -> 200


# Preprocessing



In [6]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.cluster import MiniBatchKMeans

class HOGExtractor:
  def __init__(self, img_size=(128,128)):
    self.img_size = img_size

  def extract(self, img):
    img = cv2.resize(img, self.img_size)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    return hog(
          gray,
          orientations=9,
          pixels_per_cell=(8,8),
          cells_per_block=(1,1)
      )

class BoVWExtractor:
  def __init__(self, img_size=(128,128), num_clusters=100):
    self.img_size = img_size
    self.sift = cv2.SIFT_create()
    self.kmeans = None
    self.num_clusters = num_clusters
    self.is_bovw = True

  def extract_descriptors(self, img):
    img = cv2.resize(img, self.img_size)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, desc = self.sift.detectAndCompute(gray, None)
    return desc

  def build_vocab(self, descriptors_list):
    all_desc = np.vstack([d for d in descriptors_list if d is not None])

    self.kmeans = MiniBatchKMeans(n_clusters=self.num_clusters)
    self.kmeans.fit(all_desc)

  def to_histogram(self, descriptors):
    hist = np.zeros(self.num_clusters)

    if descriptors is None:
      return hist

    clusters = self.kmeans.predict(descriptors)

    for c in clusters:
      hist[c] += 1

    return hist

class CNNExtractor:
  def __init__(self, img_size=(128,128)):
    self.img_size = img_size

  def extract(self, img):
    img = cv2.resize(img, self.img_size)
    img / 255.0
    img = np.transpose(img, (2,0,1))
    return img

In [7]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.model_selection import train_test_split
import random

random.seed(10)

class AgeDatasetPreprocessor:
  def __init__(self, base_path, extractor):
    self.base_path = base_path
    self.extractor = extractor

  def _load_labels(self, split):
    label_path = os.path.join(self.base_path, split, f"{split}_labels.txt")

    label_dict = {}

    with open(label_path, "r") as f:
      for line in f:
        filename, label = line.strip().split()
        label_dict[filename] = int(label)

    return label_dict

  def _load_dataset(self, split, limit_ratio=None):
    folder = os.path.join(self.base_path, split)
    label_dict = self._load_labels(split)

    items = list(label_dict.items())

    random.shuffle(items)

    if limit_ratio is not None:
      if limit_ratio > 1:
        limit_ratio = 1
        print("limit ratio cannot be greater than 1, defaulted to 1")
      subset_size = int(len(items) * limit_ratio)
      items = items[:subset_size]

    data = []
    labels = []

    if hasattr(self.extractor, "is_bovw"):
      descriptors_list = []

      for filename, label in items:
        path = os.path.join(folder, filename)
        img = cv2.imread(path)
        if img is None:
          continue

        desc = self.extractor.extract_descriptors(img)
        descriptors_list.append(desc)
        labels.append(label)

      if split == "train":
          self.extractor.build_vocab(descriptors_list)

      for desc in descriptors_list:
          hist = self.extractor.to_histogram(desc)
          data.append(hist)
    else:
      for filename, label in items:
        path = os.path.join(folder, filename)
        img = cv2.imread(path)
        if img is None:
          continue
        processed = self.extractor.extract(img)
        data.append(processed)
        labels.append(label)

    return np.array(data), np.array(labels)

  def load_train(self, validation_ratio=0.1, limit_ratio=None):
    X, y = self._load_dataset("train", limit_ratio)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=validation_ratio,
        random_state=42,
        stratify=y
    )

    return X_train, X_val, y_train, y_val

  def load_test(self, limit_ratio=None):
    return self._load_dataset("test", limit_ratio)

# SVM

In [8]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

class AgeSVM:
  def __init__(self, kernel='rbf', C=10, gamma='scale'):
    self.scaler = StandardScaler()
    self.model = SVC(kernel=kernel, C=C, gamma=gamma)

  def fit(self, X_train, y_train):
    X_train_scaled = self.scaler.fit_transform(X_train)
    self.model.fit(X_train_scaled, y_train)

  def predict(self, X):
    X_scaled = self.scaler.transform(X)
    return self.model.predict(X_scaled)

  def evaluate(self, X, y):
    y_pred = self.predict(X)
    acc = accuracy_score(y, y_pred)

    print("Accuracy:", acc)
    print("\nClassification Report:\n")
    print(classification_report(y, y_pred))

    return acc

# MLP

In [9]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

class AgeMLP:
  def __init__(self, hidden_layers=(256,128), max_iter=30):
    self.scaler = StandardScaler()
    self.model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation='relu',
        solver='adam',
        max_iter=max_iter,
        random_state=42
      )
  def fit(self, X_train, y_train):
    X_train_scaled = self.scaler.fit_transform(X_train)
    self.model.fit(X_train_scaled, y_train)

  def predict(self, X):
    X_scaled = self.scaler.transform(X)
    return self.model.predict(X_scaled)

  def evaluate(self, X, y):
    y_pred = self.predict(X)
    acc = accuracy_score(y, y_pred)

    print("Accuracy:", acc)
    print("\nClassification Report:\n")
    print(classification_report(y, y_pred))

    return acc

# CNN


In [10]:
import torch
from torch.utils.data import Dataset

class AgeDataset(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.long)

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [18]:
cnn_extractor = CNNExtractor()
agd = AgeDatasetPreprocessor("/content/CW_Dataset", extractor=cnn_extractor)
X_train, X_val, y_train, y_val = agd.load_train(limit_ratio=1)
X_test, y_test = agd.load_test(limit_ratio=1)

In [12]:
from torch.utils.data import DataLoader

train_loader = DataLoader(AgeDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(AgeDataset(X_val, y_val), batch_size=32)
test_loader = DataLoader(AgeDataset(X_test, y_test), batch_size=32)

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(3,32, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(32,64, kernel_size=3, padding=1)
    self.conv3 = nn.Conv2d(64,128, kernel_size=3, padding=1)

    self.pool = nn.MaxPool2d(2,2)

    self.dropout = nn.Dropout(0.5)

    self.fc1 = nn.Linear(128 * 16 * 16, 256)
    self.fc2 = nn.Linear(256, 4)

  def forward(self, x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = self.pool(F.relu(self.conv3(x)))

    x = x.view(x.size(0), -1)

    x = self.dropout(F.relu(self.fc1(x)))
    x = self.fc2(x)

    return x

In [19]:
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Assuming that we are on a CUDA machine, this should print a CUDA device:
print(device)

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)

cuda:0


In [20]:
epochs = 10

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for X_batch, y_batch in train_loader:
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

    optimiser.zero_grad()

    outputs = model(X_batch)

    loss = criterion(outputs, y_batch)

    loss.backward()

    optimiser.step()

    total_loss += loss.item()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

Epoch 1/10, Loss: 420.0111
Epoch 2/10, Loss: 171.1577
Epoch 3/10, Loss: 229.6714
Epoch 4/10, Loss: 209.3401
Epoch 5/10, Loss: 183.5796
Epoch 6/10, Loss: 161.0944
Epoch 7/10, Loss: 136.8122
Epoch 8/10, Loss: 124.9758
Epoch 9/10, Loss: 113.7237
Epoch 10/10, Loss: 113.5764


In [21]:
from sklearn.metrics import accuracy_score

def evaluate(model, loader):
  model.eval()
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for X_batch, y_batch in loader:
      X_batch = X_batch.to(device)

      outputs = model(X_batch)
      preds = torch.argmax(outputs, dim=1).cpu().numpy()

      all_preds.extend(preds)
      all_labels.extend(y_batch.numpy())

  acc = accuracy_score(all_labels, all_preds)
  print("Accuracy:", acc)
  return acc

In [22]:
print("Validation:")
evaluate(model, val_loader)

print("Test:")
evaluate(model, test_loader)

Validation:
Accuracy: 0.6676691729323309
Test:
Accuracy: 0.6517647058823529


0.6517647058823529

# TESTING

In [67]:
hog_extractor = HOGExtractor()
agd = AgeDatasetPreprocessor("/content/CW_Dataset", extractor=hog_extractor)
X_train, X_val, y_train, y_val = agd.load_train(limit_ratio=0.1)
X_test, y_test = agd.load_test(limit_ratio=0.1)

In [68]:
svm = AgeSVM()
svm.fit(X_train, y_train)

svm.evaluate(X_val, y_val)
svm.evaluate(X_test, y_test)

Accuracy: 0.6541353383458647

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.76      0.83        25
           1       0.67      0.73      0.70        56
           2       0.41      0.41      0.41        29
           3       0.68      0.65      0.67        23

    accuracy                           0.65       133
   macro avg       0.67      0.64      0.65       133
weighted avg       0.66      0.65      0.66       133

Accuracy: 0.6470588235294118

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.80      0.89        15
           1       0.62      0.94      0.75        32
           2       0.40      0.29      0.33        21
           3       0.70      0.41      0.52        17

    accuracy                           0.65        85
   macro avg       0.68      0.61      0.62        85
weighted avg       0.65      0.65      0.63        85



0.6470588235294118

In [79]:
bovw_extractor = BoVWExtractor(num_clusters=100)
pre = AgeDatasetPreprocessor("/content/CW_Dataset", extractor=bovw_extractor)

X_train, X_val, y_train, y_val = pre.load_train(limit_ratio=0.5)
X_test, y_test = pre.load_test(limit_ratio=0.5)

In [80]:
mlp = AgeMLP(max_iter=100)
mlp.fit(X_train, y_train)

mlp.evaluate(X_val, y_val)
mlp.evaluate(X_test, y_test)

Accuracy: 0.43909774436090226

Classification Report:

              precision    recall  f1-score   support

           0       0.46      0.42      0.44       120
           1       0.54      0.58      0.56       273
           2       0.36      0.29      0.32       150
           3       0.28      0.31      0.30       122

    accuracy                           0.44       665
   macro avg       0.41      0.40      0.40       665
weighted avg       0.44      0.44      0.44       665

Accuracy: 0.4447058823529412

Classification Report:

              precision    recall  f1-score   support

           0       0.45      0.41      0.43        73
           1       0.56      0.61      0.58       173
           2       0.31      0.27      0.29        96
           3       0.32      0.34      0.33        83

    accuracy                           0.44       425
   macro avg       0.41      0.41      0.41       425
weighted avg       0.44      0.44      0.44       425



0.4447058823529412